# Step 4 — 分類器訓練與評估
載入向量，依時間順序切割訓練/測試集，訓練 NB / KNN / SVM / DT / RF 並輸出混淆矩陣

Big Data & Business Analytics, National Taiwan University

In [53]:
# ── 可調整參數區 ──────────────────────────────────────
TEST_RATIO = 0.2   # 測試資料比例（後 20% 時間段）
# ──────────────────────────────────────────────────────

In [55]:
import csv
import numpy as np
from scipy import sparse
from sklearn.naive_bayes      import MultinomialNB
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.svm              import SVC
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import accuracy_score, confusion_matrix, classification_report

# [METHOD] 目前方法：時序切割 80/20 | 可替換為：移動回測（step5）
# [METHOD] 目前方法：NB/KNN/SVM/DT/RF | 可替換為：投票集成、LLM 零樣本分類

In [57]:
# 載入稀疏矩陣（由 step3 的 limit_model.npz 輸出）
X = sparse.load_npz('limit_model.npz')

# 載入對應標籤（由 step3 的 articles_processed_Ngram.csv 輸出）
y = []
with open('articles_processed.csv', newline='', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        y.append(int(row['label']))
y = np.array(y)

print(f'共 {X.shape[0]} 篇文章，{X.shape[1]} 維特徵')
print(f'看漲：{sum(y==1)} 篇，看跌：{sum(y==-1)} 篇')

共 2007 篇文章，400 維特徵
看漲：1186 篇，看跌：821 篇


In [65]:
# 依時間順序切割（articles 已按 post_time 排序）
# 不用隨機切割，保持時間先後

# split_idx = int(X.shape[0] * (1 - TEST_RATIO))
# X_train, X_test = X[:split_idx], X[split_idx:]
# y_train, y_test = y[:split_idx], y[split_idx:]

# print(f'訓練集：{X_train.shape[0]} 篇，測試集：{X_test.shape[0]} 篇')

# 隨機切割，不保持時間先後

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_RATIO,
    random_state=42,
    stratify=y  # 確保訓練/測試集的漲跌比例一致
)

print(f'訓練集：{X_train.shape[0]} 篇，測試集：{X_test.shape[0]} 篇')
print(f'訓練集看漲：{sum(y_train==1)}，看跌：{sum(y_train==-1)}')
print(f'測試集看漲：{sum(y_test==1)}，看跌：{sum(y_test==-1)}')


訓練集：1605 篇，測試集：402 篇
訓練集看漲：948，看跌：657
測試集看漲：238，看跌：164


In [66]:
# 評估函式（訓練 → 預測 → 印出混淆矩陣、準確率、precision/recall/f1）
def evaluate(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred, labels=[1, -1])

    print(f'\n===== {name} =====')
    print(f'準確率：{acc*100:.1f}%')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, labels=[1, -1],
                                target_names=['看漲', '看跌']))
    print('Confusion Matrix (列=真實, 欄=預測):')
    print(f'{"":16s} 預測漲  預測跌')
    print(f'{"真實漲":16s}  {cm[0][0]:5d}    {cm[0][1]:5d}')
    print(f'{"真實跌":16s}  {cm[1][0]:5d}    {cm[1][1]:5d}')
    return acc

In [67]:
# Naive Bayes（需要非負輸入，TF 向量天然滿足）
acc_nb = evaluate('Naive Bayes', MultinomialNB(), X_train, y_train, X_test, y_test)


===== Naive Bayes =====
準確率：61.9%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.61      1.00      0.76       238
          看跌       1.00      0.07      0.13       164

    accuracy                           0.62       402
   macro avg       0.80      0.53      0.44       402
weighted avg       0.77      0.62      0.50       402

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 238        0
真實跌                 153       11


In [68]:
# KNN（k=5）
acc_knn = evaluate('KNN (k=5)', KNeighborsClassifier(n_neighbors=5), X_train, y_train, X_test, y_test)


===== KNN (k=5) =====
準確率：68.7%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.69      0.86      0.76       238
          看跌       0.68      0.44      0.53       164

    accuracy                           0.69       402
   macro avg       0.68      0.65      0.65       402
weighted avg       0.69      0.69      0.67       402

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 204       34
真實跌                  92       72


In [69]:
# SVM（線性核）
acc_svm = evaluate('SVM', SVC(kernel='linear'), X_train, y_train, X_test, y_test)


===== SVM =====
準確率：60.7%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.60      0.99      0.75       238
          看跌       0.80      0.05      0.09       164

    accuracy                           0.61       402
   macro avg       0.70      0.52      0.42       402
weighted avg       0.68      0.61      0.48       402

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 236        2
真實跌                 156        8


In [70]:
# Decision Tree
acc_dt = evaluate('Decision Tree', DecisionTreeClassifier(random_state=42), X_train, y_train, X_test, y_test)


===== Decision Tree =====
準確率：66.9%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.70      0.76      0.73       238
          看跌       0.61      0.54      0.57       164

    accuracy                           0.67       402
   macro avg       0.66      0.65      0.65       402
weighted avg       0.66      0.67      0.67       402

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 181       57
真實跌                  76       88


In [71]:
# Random Forest
acc_rf = evaluate('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, X_test, y_test)


===== Random Forest =====
準確率：76.6%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.76      0.89      0.82       238
          看跌       0.79      0.58      0.67       164

    accuracy                           0.77       402
   macro avg       0.77      0.74      0.74       402
weighted avg       0.77      0.77      0.76       402

Confusion Matrix (列=真實, 欄=預測):
                 預測漲  預測跌
真實漲                 213       25
真實跌                  69       95


In [72]:
# 準確率彙整
print('\n===== 準確率彙整 =====')
results = [
    ('Naive Bayes',   acc_nb),
    ('KNN (k=5)',     acc_knn),
    ('SVM',           acc_svm),
    ('Decision Tree', acc_dt),
    ('Random Forest', acc_rf),
]
for name, acc in sorted(results, key=lambda x: x[1], reverse=True):
    print(f'  {name:20s}: {acc*100:.1f}%')


===== 準確率彙整 =====
  Random Forest       : 76.6%
  KNN (k=5)           : 68.7%
  Decision Tree       : 66.9%
  Naive Bayes         : 61.9%
  SVM                 : 60.7%
